In [3]:
!pip install --upgrade google-cloud-bigquery

In [4]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


In [21]:
#!pip install db-dtypes pandas-gbq
#!pip install db-dtypes
#!pip install google-cloud-bigquery-storage pyarrow

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project= 'moonlit-parsec-485600-p0')

# This query stacks the tables from different dates and unnest the stacked tables
sql = """
SELECT 
    fullVisitorId,
    date,
    p.v2ProductName, 
    p.v2ProductCategory
FROM
    `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
    UNNEST(hits) AS h,
    UNNEST(h.product) AS p
WHERE
    _TABLE_SUFFIX BETWEEN '20170101' AND '20170630'
    AND h.transaction.transactionId is NOT NULL
"""

df = client.query(sql).to_dataframe()
print(f"Successfully pulled {len(df):,} rows!")
df.head()


C:\Users\caitc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\auth\_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Successfully pulled 32,629 rows!


,fullVisitorId,date,v2ProductName,v2ProductCategory
0,0366796863244619267,20170226,Four Color Retractable Pen,Office
1,0366796863244619267,20170226,Four Color Retractable Pen,Office
2,4995606827430436569,20170226,Softsided Travel Pouch Set,Bags
3,4995606827430436569,20170226,Softsided Travel Pouch Set,Bags
4,0015950283479889703,20170226,Recycled Mouse Pad,Electronics


In [23]:
df = df.rename(columns = {
    'fullVisitorId' : 'user_id',
    'v2ProductName': 'product_name',
    'v2ProductCategory': 'product_category'
})

#df.info()

agg_df = df.groupby(['user_id','product_name']).size().reset_index(name = 'interaction_count')
agg_df

,user_id,product_name,interaction_count
0,0000213131142648941,BLM Sweatshirt,2
1,0003961110741104601,Google Laptop and Cell Phone Stickers,2
2,0003961110741104601,YouTube Custom Decals,2
3,0007933257389091624,BLM Sweatshirt,2
4,0012561433643490595,Google Kick Ball,2
...,...,...,...
13947,9990183617359422098,Windup Android,2
13948,9990797196896345494,Google Car Clip Phone Holder,2
13949,9990797196896345494,Metal Texture Roller Pen,2
13950,9990797196896345494,Switch Tone Color Crayon Pen,2


In [24]:
import os 
print(os.getcwd())

C:\Users\caitc\Senior Capstone
